# Notebook 06 — Train, Tune, Evaluate

Train a baseline and candidate models, tune using the validation set, and evaluate the selected model once on the test set.

Because late deliveries are the minority class, F1, precision, recall, and PR-AUC are used in addition to accuracy.

## 1. Imports and paths

In [1]:
from pathlib import Path
import json
import joblib
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path.cwd().parent

FEATURE_DIR = PROJECT_ROOT / "artifacts" / "05_features"
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "06_model"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load feature tables

In [3]:
train = pd.read_parquet(FEATURE_DIR / "train_features.parquet")
val = pd.read_parquet(FEATURE_DIR / "validation_features.parquet")
test = pd.read_parquet(FEATURE_DIR / "test_features.parquet")

X_train = train.drop(columns="is_late")
y_train = train["is_late"]

X_val = val.drop(columns="is_late")
y_val = val["is_late"]

# Do NOT prepare/use test yet

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (67529, 52)
Validation: (14470, 52)


## 3. Evaluation function

In [4]:
def evaluate_model(model, X, y):
    pred = model.predict(X)
    prob = model.predict_proba(X)[:, 1]

    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "pr_auc": average_precision_score(y, prob)
    }

## 4. Baseline

In [5]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)

baseline_results = evaluate_model(
    baseline,
    X_val,
    y_val
)

baseline_results

{'accuracy': 0.9465791292328957,
 'precision': 0.0,
 'recall': 0.0,
 'f1': 0.0,
 'pr_auc': 0.053420870767104355}

## 5. Simple Logistic Regression

In [6]:
logistic = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logistic.fit(X_train, y_train)

logistic_results = evaluate_model(
    logistic,
    X_val,
    y_val
)

logistic_results

{'accuracy': 0.6821700069108501,
 'precision': 0.11117886178861788,
 'recall': 0.7076326002587322,
 'f1': 0.19216581767082383,
 'pr_auc': 0.14694786196132778}

## 6. Random Forest

In [8]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_results = evaluate_model(
    rf,
    X_val,
    y_val
)

rf_results

{'accuracy': 0.9457498272287491,
 'precision': 0.16666666666666666,
 'recall': 0.0038809831824062097,
 'f1': 0.007585335018963337,
 'pr_auc': 0.1170131285441106}

## 7. XGBoost

In [10]:
from xgboost import XGBClassifier

neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

xgb_results = evaluate_model(
    xgb,
    X_val,
    y_val
)

xgb_results

{'accuracy': 0.8919834139599171,
 'precision': 0.14221014492753623,
 'recall': 0.20310478654592498,
 'f1': 0.16728822589238146,
 'pr_auc': 0.12548764338443366}

## 8. HistGradientBoosting

In [11]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

hist_gb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    random_state=42
)

hist_gb.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

hist_results = evaluate_model(
    hist_gb,
    X_val,
    y_val
)

hist_results

{'accuracy': 0.8808569454042847,
 'precision': 0.1400454201362604,
 'recall': 0.23932729624838292,
 'f1': 0.1766953199617956,
 'pr_auc': 0.12240505581149858}

## 9. Lightgbm

In [13]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm.fit(X_train, y_train)

lgbm_results = evaluate_model(
    lgbm,
    X_val,
    y_val
)

lgbm_results

{'accuracy': 0.8856254319281271,
 'precision': 0.14146341463414633,
 'recall': 0.22509702457956016,
 'f1': 0.17373939091362955,
 'pr_auc': 0.12637055111202491}

## 10. SVM

In [15]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    class_weight="balanced",
    random_state=42
)

svm.fit(X_train, y_train)

svm_pred = svm.predict(X_val)

svm_results = {
    "accuracy": accuracy_score(y_val, svm_pred),
    "precision": precision_score(y_val, svm_pred, zero_division=0),
    "recall": recall_score(y_val, svm_pred, zero_division=0),
    "f1": f1_score(y_val, svm_pred, zero_division=0)
}

svm_results

{'accuracy': 0.6884588804422944,
 'precision': 0.1105318039624609,
 'recall': 0.685640362225097,
 'f1': 0.1903735632183908}

## Comparison

In [16]:
results = pd.DataFrame({
    "Baseline": baseline_results,
    "Logistic Regression": logistic_results,
    "Random Forest": rf_results,
    "XGBoost": xgb_results,
    "HistGradientBoosting": hist_results,
    "LightGBM": lgbm_results,
}).T

display(
    results.sort_values(
        "f1",
        ascending=False
    ).round(4)
)

,accuracy,precision,recall,f1,pr_auc
Logistic Regression,0.6822,0.1112,0.7076,0.1922,0.1469
HistGradientBoosting,0.8809,0.1400,0.2393,0.1767,0.1224
LightGBM,0.8856,0.1415,0.2251,0.1737,0.1264
XGBoost,0.8920,0.1422,0.2031,0.1673,0.1255
Random Forest,0.9457,0.1667,0.0039,0.0076,0.1170
Baseline,0.9466,0.0000,0.0000,0.0000,0.0534


## Fine-tuning XGBoost

In [18]:
xgb_configs = [
    {
        "n_estimators": 200,
        "max_depth": 3,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    },
    {
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 1.0
    }
]

In [19]:
xgb_tuning_results = []

for config in xgb_configs:

    model = XGBClassifier(
        **config,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    metrics = evaluate_model(
        model,
        X_val,
        y_val
    )

    xgb_tuning_results.append({
        **config,
        **metrics
    })

xgb_tuning_df = pd.DataFrame(xgb_tuning_results)

display(
    xgb_tuning_df
    .sort_values("f1", ascending=False)
    .round(4)
)

,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,accuracy,precision,recall,f1,pr_auc
2,400,5,0.03,0.8,0.8,0.8894,0.1568,0.2445,0.1911,0.1312
1,300,4,0.05,0.9,0.9,0.8813,0.1440,0.2471,0.1820,0.1239
3,300,6,0.05,1.0,1.0,0.8920,0.1422,0.2031,0.1673,0.1255
0,200,3,0.05,0.8,0.8,0.9002,0.1509,0.1876,0.1672,0.1189


In [20]:
best_row = xgb_tuning_df.sort_values(
    "f1",
    ascending=False
).iloc[0]

best_params = {
    "n_estimators": int(best_row["n_estimators"]),
    "max_depth": int(best_row["max_depth"]),
    "learning_rate": best_row["learning_rate"],
    "subsample": best_row["subsample"],
    "colsample_bytree": best_row["colsample_bytree"]
}

In [21]:
best_xgb = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,np.float64(0.8)
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [22]:
val_prob = best_xgb.predict_proba(X_val)[:, 1]

In [23]:
threshold_results = []

for threshold in [
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50
]:
    val_pred = (val_prob >= threshold).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(
            y_val,
            val_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            val_pred,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            val_pred,
            zero_division=0
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(
    threshold_df
    .sort_values("f1", ascending=False)
    .round(4)
)

,threshold,precision,recall,f1
4,0.40,0.1294,0.4476,0.2008
5,0.45,0.1386,0.3286,0.1950
6,0.50,0.1568,0.2445,0.1911
3,0.35,0.1115,0.5666,0.1864
2,0.30,0.0954,0.6805,0.1674
1,0.25,0.0838,0.8150,0.1519
0,0.20,0.0739,0.8978,0.1366


In [24]:
best_threshold = (
    threshold_df
    .sort_values("f1", ascending=False)
    .iloc[0]["threshold"]
)

print("Best threshold:", best_threshold)

Best threshold: 0.4


In [25]:
X_test = test.drop(columns="is_late")
y_test = test["is_late"]

test_prob = best_xgb.predict_proba(X_test)[:, 1]

test_pred = (
    test_prob >= best_threshold
).astype(int)

In [1]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "xgboost",
    "joblib",
    "pyarrow",
    "SQLAlchemy",
    "psycopg2-binary",
    "python-dotenv",
]

for package in packages:
    try:
        print(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

Python: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
numpy==2.5.2
pandas==3.0.5
scipy==1.18.1
scikit-learn==1.9.0
xgboost==3.4.1
joblib==1.6.0
pyarrow==25.0.1
SQLAlchemy==2.0.52
psycopg2-binary==2.9.12
python-dotenv==1.2.3


In [1]:
from importlib.metadata import version, PackageNotFoundError

for package in [
    "notebook",
    "ipykernel",
    "matplotlib",
    "lightgbm",
    "pytest",
    "ruff",
    "pre-commit",
]:
    try:
        print(f"{package}=={version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

notebook==7.6.2
ipykernel==7.3.0
matplotlib==3.11.1
lightgbm==4.7.0
pytest: not installed
ruff: not installed
pre-commit: not installed
